In [1]:
import os
import re
import json
from dotenv import load_dotenv

from langchain_core.messages import HumanMessage
from langsmith import Client
from langsmith.evaluation import evaluate
from langsmith.schemas import Example, Run
import google.generativeai as genai

from graph import mobilePlans_agent

load_dotenv()

/home/dhanoojr/agents/mobile-plans-agent/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_360075/2566646179.py:10: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


✅ LangSmith tracing enabled


True

In [2]:
# -----------------------------
# Initialize clients
# -----------------------------
langsmith_client = Client(api_key= os.getenv("LANGSMITH_API_KEY"))
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# Test connection
print(list(langsmith_client.list_datasets()))

[Dataset(name='dialog-plan-dataset', description='Dialog broadband plan evaluation dataset', data_type=<DataType.kv: 'kv'>, id=UUID('556997d0-11b0-4fdc-8198-43d2ca12bf73'), created_at=datetime.datetime(2026, 2, 14, 17, 16, 29, 582298, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 2, 14, 17, 16, 29, 582298, tzinfo=TzInfo(0)), example_count=3, session_count=45, last_session_start_time=datetime.datetime(2026, 2, 15, 6, 43, 26, 720900), inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'Linux-6.17.0-14-generic-x86_64-with-glibc2.39', 'sdk_version': '0.7.3', 'runtime_version': '3.12.3', 'langchain_version': None, 'py_implementation': 'CPython', 'langchain_core_version': '1.2.12'}})]


In [4]:

# -----------------------------
# Example dataset
# -----------------------------
dataset_name = "dialog-plan-dataset"

# Create dataset if not exists
datasets = [d.name for d in langsmith_client.list_datasets()]
if dataset_name not in datasets:
    dataset = langsmith_client.create_dataset(
        dataset_name=dataset_name,
        description="Dialog broadband plan evaluation dataset"
    )

    examples = [
        {"input": {"question": "cheap data plan"}, "output": {"answer": "Power Plan 1300"}},
        {"input": {"question": "unlimited data plan"}, "output": {"answer": "Power Plan 2100"}},
        {"input": {"question": "30 day validity plans"}, "output": {"answer": "Power Plan 1300"}}
    ]
    for ex in examples:
        langsmith_client.create_example(
            inputs=ex["input"],
            outputs=ex["output"],
            dataset_id=dataset.id
        )

else:
    dataset = langsmith_client.read_dataset(dataset_name=dataset_name)

print("Dataset ready:", dataset.name)



Dataset ready: dialog-plan-dataset


In [5]:
# -----------------------------
# Evaluator function
# -----------------------------
def accuracy_evaluator(run: Run, example: Example):
    predicted = run.outputs.get("answer", "") 
    print(predicted["messages"][-1].content)
    clean_predicted = predicted["messages"][-1].content.lower()
    expected = example.outputs.get("answer", "").lower()

    score = 1 if expected in clean_predicted else 0

    return {
        "key": "accuracy",
        "score": score
    }

# --

In [6]:
# -----------------------------
# Your system under test
# -----------------------------

def my_retrieval_system(inputs: dict):
    question = inputs
    model = mobilePlans_agent()
    config={"configurable": {"thread_id": "langsmith-eval-thread"}}

    response = model.invoke({"messages":HumanMessage(content=inputs["question"])},config=config)

    return {
        "answer": response
    }

In [7]:
# -----------------------------
# Run evaluation experiment
# -----------------------------
experiment_results = evaluate(
    my_retrieval_system,
    data=dataset,
    evaluators=[accuracy_evaluator],
    experiment_prefix="dialog-plan-eval"
)


View the evaluation results for experiment: 'dialog-plan-eval-f606b9d8' at:
https://smith.langchain.com/o/c9f68f42-9218-4b4d-9933-a914ac0df2cc/datasets/556997d0-11b0-4fdc-8198-43d2ca12bf73/compare?selectedSessions=eec20c4b-ca55-4208-92b1-e543aeb4b727




0it [00:00, ?it/s]

🔧 Tool called with query: 30 day validity plans


1it [00:14, 14.52s/it]

Of course! I can certainly help you with that.

Based on your query, I have used the `retrieve_plans` tool to find the best postpaid options for you. Here are the most relevant packages available:

For individual use, our **Power Plans** are very popular:

*   **Power Plan 1300:** This is a great value option at **Rs. 1300.00**. It comes with **Unlimited Calls** to any network and **20GB of data**.
*   **Power Plan 2100:** If you need more data, this plan offers **50GB** and also includes **Unlimited Calls** to any network for **Rs. 2100.00**.

We also have some excellent plans for specific groups:

*   **Family Plans:** Perfect for keeping your whole family connected. These plans offer shared data benefits and special rates for up to 5 members.
*   **Dialog Prashansa Plans:** We have exclusive plans with special rates and premium benefits specifically for Government Pensioners.
*   **Friend Circle:** You can get lifetime discounts with your friends through this special program.

All t

2it [00:29, 14.88s/it]

Of course! I can certainly help you with that.

To find the best options for you, I've used our `retrieve_plans` tool. Based on the information, here are some of our most popular postpaid packages that might suit your needs.

I'll start with our most relatable individual plans:

**1. Power Plan 1300**
This is a fantastic and affordable all-around plan.
*   **Price:** Rs. 1300.00
*   **Data:** 20GB (with Data Rollover & Sharing)
*   **Calls:** Unlimited to any network
*   **Key Perk:** Includes a free ViU+ Subscription for 12 months.

**2. Power Plan 2100**
If you're a heavy data user and love entertainment, this premium plan is perfect.
*   **Price:** Rs. 2100.00
*   **Data:** 50GB (with Data Rollover & Sharing)
*   **Calls:** Unlimited to any network
*   **Key Perks:** Includes free ViU+ and Lionsgate Play subscriptions for 12 months.

We also have some special plans for specific needs:

*   **Family Plans:** Perfect for keeping your family connected. These plans offer shared data ben

3it [00:44, 14.91s/it]

Of course! I can certainly help you with that.

Based on your query, I've used my `retrieve_plans` tool to find the most suitable postpaid packages for you. Here are the best options I found:

For a great balance of data and value, I would recommend starting with our Power Plans:

*   **Power Plan 1300**: This is a very popular and affordable option. For just **Rs. 1300.00**, you get **Unlimited Calls** to any network and **20GB of data**. It's perfect for everyday use.
*   **Power Plan 2100**: If you're a heavy data user, this premium plan is a great fit. For **Rs. 2100.00**, it includes **Unlimited Calls** and a massive **50GB of data**, plus entertainment subscriptions.

We also have some more specialized plans you might be interested in:

*   **Family Plans**: Ideal for keeping your whole family connected. These plans offer shared data benefits and special rates for up to 5 members.
*   **Smartphone Plan**: If you're looking to get a new phone, this plan allows you to get the lates

3it [00:45, 15.04s/it]


In [10]:
print("Experiment completed!")
print(experiment_results)

Experiment completed!
<ExperimentResults dialog-plan-eval-f606b9d8>
